# AIODOO Colab — Train, Validate, Package

Thin orchestration only:

- **Training** runs exclusively through **aiodoo-training**.
- **Validation** runs exclusively through **aiodoo-validation**.
- **Model packaging** runs exclusively through **aiodoo-model**.

This notebook owns only experiment selection/configuration, progress display,
logging, checkpoint monitoring, artifact browsing, resume workflow, and
Google Drive integration — never the algorithms themselves.

Set `AIODOO_COLAB_ROOT` to the cloned `aiodoo-colab` repository path, then run cells in order.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

# Notebook is responsible for path setup (no pip install of aiodoo-colab).
COLAB_ROOT = Path(os.environ.get("AIODOO_COLAB_ROOT", ".")).resolve()
COLAB_ROOT = Path("/content/aiodoo-colab").resolve()
PYTHON_DIR = COLAB_ROOT / "python"
if str(PYTHON_DIR) not in sys.path:
    sys.path.insert(0, str(PYTHON_DIR))

print("aiodoo-colab root:", COLAB_ROOT)



In [ ]:
from artifacts import browse_training_artifacts, summarize_artifacts
from colab_logging import configure_logging, get_logger
from config import load_config
from experiments import ExperimentStore
from models import ModelStore
from model_packaging import ModelRegistry, publish_adapter, summarize_packaging
from repository import TrainingRepository
from trainer import build_training_context, run_training, summarize_result
from training_ui import TrainingMonitor
from validation import run_validation, summarize_validation
from workspace import prepare_workspace

configure_logging()
logger = get_logger()
logger.info("Modules imported")


In [ ]:
# Mount Drive + prepare AIODOO workspace layout
config = load_config()
workspace = prepare_workspace(config)
print("Workspace:", workspace.root)

In [ ]:
# Ensure frozen aiodoo-training checkout
repo = TrainingRepository.from_workspace(workspace, config)
if repo.exists():
    repo.update()
else:
    repo.clone()
repo.verify()
print("Training repository:", repo.path)

In [ ]:
# Load training config (read-only). Change TRAINING_ID as needed.
TRAINING_ID = "coding"

experiments = ExperimentStore(workspace=workspace)
experiment = experiments.load(TRAINING_ID)
print("Training:", experiment.training_id)
print("Model id:", experiment.model_id)
print("Dataset version:", experiment.dataset_version)

In [ ]:
# Ensure Hugging Face base model is available under workspace.model_cache
# (Colab local SSD: /content/aiodoo-model-cache — not Google Drive).
assert isinstance(experiment.model_id, str) and experiment.model_id.strip()
model_store = ModelStore(workspace=workspace, model_id=experiment.model_id)
model_path = model_store.ensure()
print("Model path:", model_path)

In [ ]:
# Build context and invoke aiodoo-training public entrypoint (train.py).
# auto_resume=True: if a Drive checkpoint already exists for this training id
# (e.g. resuming after a Colab disconnect), aiodoo-colab locates the latest
# one and injects `checkpointing.resume_from` into a scratch copy of the
# training config — aiodoo-training's own ResumeCoordinator does everything
# else. Set auto_resume=False for an intentional from-scratch run.
context = build_training_context(workspace, experiment, model_path=model_path)

monitor = TrainingMonitor(
    training_id=experiment.training_id,
    model_name=experiment.model_id,
    dataset_version=experiment.dataset_version,
)
monitor.display()
result = run_training(context, auto_resume=True, on_log_line=monitor.on_line)
monitor.finish(result)
print(summarize_result(result))

## Checkpoint / artifact browsing (read-only)

Everything below is orchestration over Drive paths already produced by
aiodoo-training — this notebook never re-derives or writes them itself (see
`workspace.Workspace.checkpoints_root` / `artifacts.browse_training_artifacts`).

In [ ]:
# Discover every checkpoint / adapter / merged / export artifact for this run.
run_artifacts = browse_training_artifacts(workspace, experiment.training_id)
print(summarize_artifacts(run_artifacts))

## Validation — exclusively via aiodoo-validation

Certification, scoring, and reporting all happen inside
`aiodoo_validation.api.ValidationService`; this notebook only supplies the
paths training just produced and prints the result (see
`aiodoo-validation/docs/integration.md`'s Colab example).

In [ ]:
# Requires `pip install -e ../aiodoo-validation` (or its repo on sys.path).
# Fails closed if `result.success` is False — never certifies a failed run.
validation_outcome = run_validation(context, result, execution_tier="standard")
print(summarize_validation(validation_outcome))

## Model packaging — exclusively via aiodoo-model

Adapter composition, fingerprinting, metadata, and storage are all
`aiodoo_model.publishing.PublishingService`; this notebook only points at the
aiodoo-training output directory and the Drive-backed registry/storage roots
(see `workspace.Workspace.model_registry` / `model_registry_storage`).
Publishing is idempotent — re-running this cell after a disconnect reports
`already_published=True` instead of failing.

In [ ]:
# Requires `pip install -e ../aiodoo-model` in the Colab runtime.
registry = ModelRegistry.from_workspace(workspace)
packaging_result = publish_adapter(registry, experiment, result)
print(summarize_packaging(packaging_result))